# Day 13 — Async Python: asyncio, async/await, httpx & Concurrency Limits (6 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: understand what really happens while a program waits for a network reply, and use that waiting time to run many requests at once, without breaking a customer's rate limit, losing the order of the results or hiding errors.

- Deliverable: day13/async_export.py, a concurrent CMS exporter that writes the same CSV as the sequential sync_export.py in less than half the time, with at most 3 requests in flight, polite retries and the 8 given tests green. Plus day13/dsa.py.

- Where this sits: Week 2, Phase 1 (Foundation), day 6 of 7. FastAPI (Week 4) is an async framework, the Anthropic SDK has an async client, and every RAG pipeline and agent you build from Week 5 calls slow services in parallel. Today is the base for all of that.

Time split (5h):

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 6, running every example |
| Hands-on | 2.5h | async_export.py: fetch_page, fetch_all_entries, export, main |
| DSA | 0.5h | Max requests in flight, first rate-limit violation |
| Revision | 0.5h | Quiz, Remember cells in the notebook, a clean commit |

Open items first: GitHub still shows only day01/ to day04/, and Day 4's files are still the starters. Days 5 to 12 are not pushed. If that work is on your laptop, push it before you start, one branch per day. Today's folder is self-contained (it ships its own fake CMS, a sequential exporter and the tests), so you can do it even if earlier days are unfinished. Tomorrow's weekly score reads the repo.

## Recap (2 minutes, from memory)

Answer out loud or on paper before you look anything up. The answers are at the end.

- Your pyproject.toml says httpx>=0.27, and the project also has a uv.lock. What does each file promise, and what does uv sync --locked do in a customer's CI?

- Why should a module inside your package call logging.getLogger(__name__) and never logging.basicConfig()?

- ruff and mypy both run before a reviewer reads your code. Name one bug each of them catches that the other one cannot.

## Concept 1: Waiting is the bottleneck, and the event loop

### Why it matters for an FDE

Almost everything an FDE builds talks to slow things: a customer's CMS, a CRM, a vector database, the Claude API. A single LLM call can take 5 to 20 seconds. If your code waits for each reply before sending the next request, a job that could take 10 seconds takes 3 minutes, and the customer's demo stalls in front of their VP.

The fix is not a faster computer. During a network call your CPU does almost nothing: it sends a few bytes and then waits. Async code uses that waiting time to start or finish other requests.

### How it works

Two words are easy to mix up. Concurrency means several tasks are in progress at the same time, taking turns. Parallelism means several tasks are computed at the same instant, on several CPU cores. asyncio gives you concurrency on one thread. That is exactly right for work that waits on the network or disk (I/O-bound work), and it does nothing for work that keeps the CPU busy (CPU-bound work, such as resizing images).

At the centre is the event loop: one loop that keeps a list of tasks. It runs one task until that task says "I am waiting now" (that is what await means), then it runs another task that is ready. When a network reply arrives, the waiting task becomes ready again. You know this model already: Node.js runs your JavaScript on one thread with an event loop, and fetch() returns a Promise instead of blocking. Python works the same way, with one difference you will meet in Concept 2: in Python you start the loop yourself.

In [ ]:
# Waiting one by one vs waiting together: three slow "API calls".
import asyncio
import time


def fetch_blocking(page: int) -> str:
    time.sleep(1)  # pretend the CMS takes 1 s to answer
    return f"page {page}"


async def fetch_async(page: int) -> str:
    await asyncio.sleep(1)  # same 1 s wait, but the loop can run others
    return f"page {page}"


async def main() -> None:
    start = time.perf_counter()
    results = await asyncio.gather(fetch_async(1), fetch_async(2),
                                   fetch_async(3))
    print(results, f"{time.perf_counter() - start:.1f}s")


start = time.perf_counter()
print([fetch_blocking(p) for p in (1, 2, 3)],
      f"{time.perf_counter() - start:.1f}s")
# -> ['page 1', 'page 2', 'page 3'] 3.0s   (1 + 1 + 1)
asyncio.run(main())
# -> ['page 1', 'page 2', 'page 3'] 1.0s   (the three waits overlap)

The blocking version adds the three waits together. The async version starts all three, and the loop waits for them at the same time, so the total is the longest single wait. asyncio.gather() is Python's Promise.all().

### Real-world FDE example

A customer's Contentstack stack has 4,000 entries, returned 100 per page with about 1.5 seconds per page. Sequential: 40 pages x 1.5 s = 60 s. With 4 pages in flight: about 15 s. The same idea applies to an embedding job that sends 2,000 chunks to an embeddings API in batches: concurrency turns an overnight script into a coffee break.

### Common mistakes

- Expecting async to speed up CPU-heavy work. Parsing a 2 GB JSON file or running a model locally keeps the one thread busy, so nothing else can run. That needs threads, processes or a different design.

- Thinking "async" means "in the background". A task only gives the loop a chance to run other tasks when it reaches an await. Code between two awaits runs without interruption.

## Concept 2: Coroutines, async def and await

### Why it matters for an FDE

FastAPI route handlers, the Anthropic SDK's AsyncAnthropic client, MCP servers and the Claude Agent SDK are all written with async def and await. You will read and write these two keywords every day from Week 4. Getting the basic rules wrong produces bugs that raise no error at all: the work simply does not happen.

### How it works

A function defined with async def is a coroutine function. Calling it does not run its body. It returns a coroutine object, a paused piece of work. await runs that work and gives you its result. You may only write await inside an async def.

In [ ]:
# A coroutine function returns a coroutine object; await runs it.
import asyncio


async def get_title(uid: str) -> str:
    print(f"  fetching {uid}")
    await asyncio.sleep(0.1)
    return f"title of {uid}"


async def main() -> None:
    c = get_title("blt001")  # nothing runs yet, no "fetching" line
    print(type(c).__name__)  # -> coroutine
    title = await c  # NOW it runs
    # -> "  fetching blt001" is printed at this point
    print(title)  # -> title of blt001


asyncio.run(main())

This is the first real difference from JavaScript. In JS, calling an async function starts it immediately and returns a Promise that is already running. In Python, calling it only creates the coroutine; nothing happens until something awaits it or schedules it as a task (Concept 3).

So how does the very first coroutine start, if await is only allowed inside other coroutines? With asyncio.run(main()). It creates an event loop, runs main() until it finishes, closes the loop and returns main's result. Call it once, at the entry point of your program, the same place where Day 12 put configure_logging().

In [ ]:
# Forgetting await: no error at the call, just a warning and no work done.
import asyncio


async def save_entry(uid: str) -> None:
    await asyncio.sleep(0.1)
    print(f"saved {uid}")


async def main() -> None:
    save_entry("blt001")  # GOTCHA: missing await, the entry is NOT saved
    # -> RuntimeWarning: coroutine 'save_entry' was never awaited
    await save_entry("blt002")  # -> saved blt002


asyncio.run(main())
# Only blt002 is saved. Python warns, but the program does not crash.

Forgetting await gives a RuntimeWarning and no error. In a real exporter, that is an entry that silently was not saved. The warning is your only clue, so read warnings in your logs.

In [ ]:
# Two awaits in a row still run one after the other.
import asyncio
import time


async def fetch(page: int) -> int:
    await asyncio.sleep(1)
    return page


async def main() -> None:
    start = time.perf_counter()
    a = await fetch(1)  # waits 1 s here...
    b = await fetch(2)  # ...then another 1 s here
    print(a, b, f"{time.perf_counter() - start:.1f}s")  # -> 1 2 2.0s


asyncio.run(main())

The other common surprise: await means "wait for this, then continue". Two awaits in a row run one after the other, exactly like synchronous code. Writing async does not make anything concurrent by itself. You need gather or tasks for that.

GOTCHA: in a Jupyter notebook an event loop is already running, so asyncio.run() raises RuntimeError: asyncio.run() cannot be called from a running event loop. In a notebook cell you write await main() directly. In a .py script you use asyncio.run(main()). Today's notebook uses the first form, your scripts the second.

### Real-world FDE example

A customer's FastAPI service has a webhook endpoint that receives "entry published" events from Contentstack. The handler is async def, and inside it you await a call to re-embed the entry. If someone writes reindex(entry) without await, the endpoint returns 200, Contentstack is happy, and the search index silently stops updating. You find it only through the RuntimeWarning in the logs.

### Common mistakes

- Calling a coroutine function without await (or without making a task from it). The work never runs.

- Calling asyncio.run() inside code that is already async, for example inside a FastAPI handler or a notebook. There is already a loop: just await.

- Making every function async "just in case". Only functions that await something need async def. A plain helper such as entry_to_row() stays a normal function.

## Concept 3: Running coroutines together: gather, create_task and TaskGroup

### Why it matters for an FDE

Concurrency is where the speed comes from, and also where results get mixed up and errors get lost. A customer cares about two things: every page arrives, and the rows come out in the right order. These three tools decide both.

### How it works

asyncio.gather(c1, c2, c3) runs the coroutines concurrently and returns a list of their results in the order you passed them, not the order they finished. That is what keeps pages in order without any sorting.

In [ ]:
# gather() returns results in the order you passed them, not finish order.
import asyncio
import time


async def fetch(page: int, seconds: float) -> str:
    await asyncio.sleep(seconds)
    print(f"  finished page {page}")
    return f"page {page}"


async def main() -> None:
    start = time.perf_counter()
    results = await asyncio.gather(fetch(1, 0.3), fetch(2, 0.1),
                                   fetch(3, 0.2))
    # -> finished page 2, then page 3, then page 1 (fastest first)
    print(results)  # -> ['page 1', 'page 2', 'page 3']  (argument order)
    print(f"{time.perf_counter() - start:.1f}s")  # -> 0.3s (the slowest)


asyncio.run(main())

asyncio.create_task(coro) wraps a coroutine in a task: it is scheduled on the loop and starts running as soon as the current code reaches an await. You collect the result later with await task. A task is the closest thing in Python to a JavaScript Promise that is already running.

In [ ]:
# create_task starts work now; await collects the result later.
import asyncio
import time


async def fetch(page: int) -> str:
    await asyncio.sleep(1)
    return f"page {page}"


async def main() -> None:
    start = time.perf_counter()
    t1 = asyncio.create_task(fetch(1))  # scheduled, starts at next await
    t2 = asyncio.create_task(fetch(2))
    print("both started")  # -> both started   (printed immediately)
    a = await t1  # both run while we wait here
    b = await t2  # already done, returns at once
    print(a, b, f"{time.perf_counter() - start:.1f}s")
    # -> page 1 page 2 1.0s


asyncio.run(main())

asyncio.TaskGroup (Python 3.11+) is the modern, safest way to run a set of tasks. You create tasks inside an async with block, and the block does not end until all of them are done. If one task fails, the group cancels the others and raises the errors together as an ExceptionGroup, which you catch with except*. This is called structured concurrency: no task can outlive the block that started it, so nothing keeps running unnoticed after an error.

In [ ]:
# TaskGroup: start many tasks; leaving the block waits for all of them.
import asyncio


async def fetch(skip: int) -> list[str]:
    if skip == 6:
        raise ValueError(f"bad page at skip={skip}")  # fails at once
    await asyncio.sleep(0.1)
    print(f"  done skip={skip}")
    return [f"e{skip}", f"e{skip + 1}", f"e{skip + 2}"]


async def main(skips: list[int]) -> None:
    try:
        async with asyncio.TaskGroup() as tg:
            tasks = [tg.create_task(fetch(s)) for s in skips]
        # here every task has finished successfully
        print([t.result() for t in tasks])
    except* ValueError as group:  # except* handles an ExceptionGroup
        for err in group.exceptions:
            print("failed:", err)


asyncio.run(main([0, 3]))
# ->   done skip=0
# ->   done skip=3
# -> [['e0', 'e1', 'e2'], ['e3', 'e4', 'e5']]
asyncio.run(main([0, 3, 6]))
# -> failed: bad page at skip=6
# no "done" lines: the group cancelled skip=0 and skip=3 for you

Which one to use: TaskGroup for a set of tasks that should all succeed (like the pages of one export). gather when you want a list of results from a few coroutines and you are fine with the default error behaviour. If you need "keep going and collect failures", gather(..., return_exceptions=True) returns exceptions as items in the result list instead of raising.

FDE TIP: with plain gather, if one page raises, gather raises that error to you, but the other requests keep running in the background. TaskGroup cancels them. For anything that writes data, prefer TaskGroup.

### Real-world FDE example

Before a migration you compare two systems: for each of 500 content types, fetch the entry count from Drupal and from Contentstack and report the differences. One TaskGroup with 1,000 small tasks (behind a limit, Concept 4) finishes in seconds. If the Contentstack token expires halfway, the group stops everything and reports one clear 401, instead of 900 more failing requests.

### Common mistakes

- Sorting the results afterwards to "fix" the order. gather and the task list you built are already in the order you created them. Sorting by uid can hide a bug where a page was fetched twice or skipped.

- Creating a task and never awaiting it or keeping a reference to it. The loop holds only a weak reference to tasks, so an unreferenced task can disappear before it finishes. Keep tasks in a list or use a TaskGroup.

- Reading task.result() inside the TaskGroup block. Results are only guaranteed after the block ends.

## Concept 4: Limits: Semaphore and timeouts

### Why it matters for an FDE

Unlimited concurrency is how you get banned from a customer's API. Contentstack, Salesforce, Zendesk and the Claude API all have rate limits, and many enterprise systems also limit concurrent connections per key. Going over the limit returns HTTP 429, and repeated abuse can get the key blocked during the customer's working day. A good FDE can say exactly how many requests their integration has open at any moment.

### How it works

An asyncio.Semaphore(3) is a counter with 3 slots. async with sem: takes a slot, or waits until one is free, and gives the slot back when the block ends, even if an error happens inside. Think of it as a bouncer at a door who lets in three people at a time.

In [ ]:
# A Semaphore lets at most N coroutines into a block at the same time.
import asyncio
import time

stats = {"now": 0, "peak": 0}  # a dict, so the function can update it


async def fetch(sem: asyncio.Semaphore, page: int) -> int:
    async with sem:  # wait here until one of the N slots is free
        stats["now"] += 1
        stats["peak"] = max(stats["peak"], stats["now"])
        await asyncio.sleep(0.5)  # the request itself
        stats["now"] -= 1
    return page  # the slot is released when the "async with" block ends


async def main() -> None:
    sem = asyncio.Semaphore(3)  # the customer's limit: 3 at a time
    start = time.perf_counter()
    pages = await asyncio.gather(*(fetch(sem, p) for p in range(1, 11)))
    print(pages)  # -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    print(stats["peak"], f"{time.perf_counter() - start:.1f}s")
    # -> 3 2.0s   (10 pages through 3 slots = 4 rounds of 0.5 s)


asyncio.run(main())

Ten pages, three slots: the pages go through in four rounds (3, 3, 3, 1) of 0.5 s, so 2.0 s. With no limit it would be 0.5 s on your laptop, and a pile of 429 errors on a real customer system. The second tool is a timeout. A request that never answers would otherwise hold its slot forever.

In [ ]:
# asyncio.timeout(): give up on a slow step instead of hanging forever.
import asyncio


async def slow_cms() -> str:
    await asyncio.sleep(2)  # the customer's server is stuck
    return "data"


async def main() -> None:
    try:
        async with asyncio.timeout(0.5):  # Python 3.11+
            print(await slow_cms())
    except TimeoutError:
        print("gave up after 0.5 s")  # -> gave up after 0.5 s


asyncio.run(main())

httpx has its own timeout per request (Concept 5), which is what you use for HTTP. asyncio.timeout() is for a whole step that may contain several awaits, such as "the whole export must finish within 10 minutes".

GOTCHA: hold the slot only while the request is open. If a page that got a 503 sleeps for its retry while still inside async with sem:, it blocks a slot that another page could use. Take the slot, make the request, leave the block, then sleep.

### Real-world FDE example

The Claude API limits requests and tokens per minute by usage tier. When you summarise 3,000 support tickets for a customer, a semaphore of, say, 8 plus retries that respect the retry-after header is the difference between a clean nightly job and a 429 storm that also slows down the customer's other Claude workloads on the same organisation.

### Common mistakes

- Creating the semaphore inside the function that each task runs. Every task then gets its own semaphore with three free slots, and there is no limit at all. Create it once per run and pass it in.

- Using a limit that is higher than the server's. Today's mock CMS rejects a fourth concurrent request; going higher makes the export slower, not faster (you will measure this).

- No timeout. One stuck connection then hangs the whole export until someone kills it.

## Concept 5: httpx.AsyncClient, async HTTP

### Why it matters for an FDE

requests (Day 6) is synchronous only; every call blocks the event loop. httpx has almost the same API as requests and also an async client. It is what the Anthropic and OpenAI Python SDKs use internally, so the timeouts and errors you learn here are the same ones you will see from LLM calls. The current version is 0.28.1: pip install httpx.

### How it works

Create one httpx.AsyncClient per run with async with, set base_url, shared headers and a timeout on it, and await client.get(...) for each request. The client keeps connections open and reuses them (connection pooling), which saves the TLS handshake on every request. Status codes, .json(), .headers and the 4xx-is-still-a-reply rule are the same as in requests.

Start the fake CMS first, in its own terminal: cd day13 then python mock_cms.py. It takes about 0.5 s per request, rejects a fourth concurrent request with 429 and fails a few pages once with 503.

In [ ]:
# httpx.AsyncClient: one client, three page requests at the same time.
# Start day13/mock_cms.py in another terminal first.
import asyncio
import time

import httpx


async def main() -> None:
    async with httpx.AsyncClient(
        base_url="http://127.0.0.1:8013",  # every URL below is relative
        headers={"api_key": "demo-key-123"},  # sent with every request
        timeout=5.0,  # FDE TIP: always set one; never wait forever
    ) as client:
        start = time.perf_counter()
        url = "/v3/content_types/article/entries"
        responses = await asyncio.gather(
            *(client.get(url, params={"skip": s, "limit": 3})
              for s in (0, 3, 6)))
        for r in responses:
            uids = [e["uid"] for e in r.json()["entries"]]
            print(r.status_code, uids)
        print(f"{time.perf_counter() - start:.1f}s")


asyncio.run(main())
# -> 200 ['art001', 'art002', 'art003']
# -> 200 ['art004', 'art005', 'art006']
# -> 200 ['art007', 'art008', 'art009']
# -> 0.5s   (three 0.5 s requests, overlapped)

In [ ]:
# No limit on our side: 6 requests at once, the CMS answers 429.
import asyncio

import httpx


async def main() -> None:
    async with httpx.AsyncClient(base_url="http://127.0.0.1:8013",
                                 headers={"api_key": "demo-key-123"},
                                 timeout=5.0) as client:
        url = "/v3/content_types/article/entries"
        rs = await asyncio.gather(
            *(client.get(url, params={"skip": s, "limit": 3})
              for s in range(0, 18, 3)))
        print([r.status_code for r in rs])
        print(rs[-1].headers.get("Retry-After"), rs[-1].json())


asyncio.run(main())
# -> [200, 200, 200, 429, 429, 429]   (which ones get 429 can vary)
# -> 1 {'error_message': 'Too many concurrent requests (limit 3)'}

That second example is the reason for Concept 4: the CMS answered three requests and rejected three. The error body and the Retry-After header tell you how long to wait.

For tests you do not need a server. httpx.MockTransport sends each request to a function you write, which returns a response. It is the async version of Day 10's faked HTTP, and today's given tests use it.

In [ ]:
# Testing async HTTP with no server: httpx.MockTransport.
import asyncio

import httpx


async def fake_cms(request: httpx.Request) -> httpx.Response:
    # every request the client makes arrives here instead of the network
    if request.headers.get("api_key") != "k":
        return httpx.Response(401, json={"error_message": "Invalid API key"})
    return httpx.Response(200, json={"entries": [{"uid": "blt001"}]})


async def main() -> None:
    transport = httpx.MockTransport(fake_cms)
    async with httpx.AsyncClient(transport=transport,
                                 base_url="http://cms.test") as client:
        r = await client.get("/entries", headers={"api_key": "k"})
        print(r.status_code, r.json())
        # -> 200 {'entries': [{'uid': 'blt001'}]}
        r = await client.get("/entries")  # no key
        print(r.status_code, r.json()["error_message"])
        # -> 401 Invalid API key


asyncio.run(main())

### Real-world FDE example

Errors map like this: a connection refused or DNS failure raises httpx.ConnectError, a slow server raises httpx.ReadTimeout, and both are subclasses of httpx.TransportError, which is the one exception you catch for "retry, the network had a bad moment". A 401 is not an exception at all: it is a response with status_code 401, and retrying it only fills the customer's audit log with failed logins.

### Common mistakes

- Creating a new AsyncClient for every request. It works, but it throws away connection pooling and can exhaust sockets on large exports. One client per run.

- Using requests.get() inside async def. It blocks the loop, so every other task stops while it waits (Concept 6 shows the effect).

- Leaving the client open. Without async with (or await client.aclose()) you get "Unclosed client" warnings and leaked connections.

## Concept 6: Patterns for real async work: fan-out pagination and blocking code

### Why it matters for an FDE

Paginated APIs are everywhere: CMS entries, CRM contacts, ticket lists, Drive files. Most real async work is "get the first page, learn how much there is, fetch the rest concurrently, keep the order". The second pattern is knowing what not to do inside async code, because one blocking call quietly removes all the benefit.

### How it works

If the API returns a total count (Contentstack does with include_count=true), you can compute every other page's skip from the first response and request them all at once. The list of skips you build is the order you want, so the results come back in that order.

In [ ]:
# Fan-out pagination: page 1 tells you the count, then ask for the rest.
import asyncio
import random

CMS = [f"art{n:03d}" for n in range(1, 11)]  # 10 fake entries


async def get_page(skip: int, limit: int) -> dict:
    await asyncio.sleep(random.uniform(0.05, 0.2))  # random finish order
    return {"entries": CMS[skip:skip + limit], "count": len(CMS)}


async def main() -> None:
    first = await get_page(0, 3)  # must come first: we need "count"
    skips = list(range(3, first["count"], 3))
    print(skips)  # -> [3, 6, 9]
    rest = await asyncio.gather(*(get_page(s, 3) for s in skips))
    pages = [first, *rest]  # same order as skips, whatever finished first
    print([uid for page in pages for uid in page["entries"]])
    # -> ['art001', 'art002', ..., 'art010']   (CMS order, every time)


asyncio.run(main())

If the API only gives you a "next page" cursor instead of a count, you cannot fan out like this, because each page tells you where the next one is. Then the pages stay sequential and concurrency moves up a level: several content types or several customers at once.

Blocking code is any call that waits without await: time.sleep(), requests.get(), a slow database driver that is not async, or heavy CPU work. While it runs, the event loop is stuck and every task waits. If you must call blocking code, asyncio.to_thread(func, *args) runs it in a worker thread and gives you something to await.

In [ ]:
# One blocking call inside async code stops EVERY task, not just its own.
import asyncio
import time


async def bad_fetch(page: int) -> int:
    time.sleep(0.5)  # GOTCHA: blocks the whole event loop
    return page


async def good_fetch(page: int) -> int:
    # run the blocking function in a worker thread, await the result
    await asyncio.to_thread(time.sleep, 0.5)
    return page


async def timed(fetch) -> None:
    start = time.perf_counter()
    await asyncio.gather(*(fetch(p) for p in range(4)))
    print(fetch.__name__, f"{time.perf_counter() - start:.1f}s")


asyncio.run(timed(bad_fetch))  # -> bad_fetch 2.0s   (4 x 0.5 s, in turn)
asyncio.run(timed(good_fetch))  # -> good_fetch 0.5s  (overlapped)

### Real-world FDE example

A customer's export job uses an async HTTP client for the CMS, but writes each page into Postgres with a synchronous driver. Under load it is no faster than the old script. The profile shows the loop blocked during every insert. Moving the inserts to asyncio.to_thread() (or switching to an async driver such as asyncpg) brings the speed back. Recognising this in someone else's code is a common FDE debugging task.

### Common mistakes

- Fanning out with a cursor-based API by guessing page numbers. You get duplicates or gaps. Read the API's pagination rules first.

- time.sleep() for a retry wait inside async code. Use await asyncio.sleep().

- Using to_thread for everything. It is a bridge for the odd blocking call, not a replacement for async libraries.

## Check your understanding (predict first, then run)

Write your prediction down for each snippet, then run it in the notebook or a scratch file. The answers are at the end.

- Six jobs of 1 second each go through asyncio.Semaphore(2) with gather. What does it print, and how long does it take?

- Inside main(), x = hello() where async def hello() -> str: return "hi". What do print(x == "hi") and then print(await x == "hi") show, and why?

- await asyncio.gather(worker("a", 0.3), worker("b", 0.1)), where each worker sleeps for its delay, prints "finished" and its name, then returns its name. In what order do the lines print, and what is the returned list?

- t = asyncio.create_task(asyncio.sleep(1)), then await asyncio.sleep(1), then await t. Is the total time about 1 s or about 2 s?

In [ ]:
# Check your understanding: four snippets. Predict, then run each main.
import asyncio
import time


# Question 1: six 1-second jobs through a Semaphore with 2 slots
async def job(n: int) -> int:
    await asyncio.sleep(1)
    return n


async def q1() -> None:
    sem = asyncio.Semaphore(2)

    async def limited(n: int) -> int:  # a coroutine defined inside q1
        async with sem:
            return await job(n)

    start = time.perf_counter()
    print(await asyncio.gather(*(limited(n) for n in range(6))))
    print(f"{time.perf_counter() - start:.1f}s")


# Question 2: what is x before and after await?
async def hello() -> str:
    return "hi"


async def q2() -> None:
    x = hello()
    print(x == "hi")
    print(await x == "hi")


# Question 3: finish order vs result order
async def worker(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    print("finished", name)
    return name


async def q3() -> None:
    print(await asyncio.gather(worker("a", 0.3), worker("b", 0.1)))


# Question 4: a task and a sleep at the same time
async def q4() -> None:
    start = time.perf_counter()
    t = asyncio.create_task(asyncio.sleep(1))
    await asyncio.sleep(1)
    await t
    print(f"{time.perf_counter() - start:.1f}s")


for question in (q1, q2, q3, q4):
    asyncio.run(question())

## Hands-on task (about 2.5 hours): day13/async_export.py

The scenario: the customer's content lead says: "The nightly export takes too long and it will be ten times bigger after the migration. Make it faster, but our CMS team says: never more than 3 requests at once per key, or they will block us."

Before you start: git switch main && git pull, then git switch -c day13. Create a venv in day13/ and pip install httpx pytest. Everything else is in the folder.

```bash
# day13/ as you get it
day13/
    mock_cms.py           # the fake customer CMS (given, do not edit)
    cms_common.py         # key, CSV, flags, logging (given)
    sync_export.py        # the sequential baseline (given, read it)
    async_export.py       # YOUR task: 4 TODO functions
    test_async_export.py  # 8 given tests (do not edit)
    dsa.py                # today's two DSA problems
    day13_notes.ipynb     # runnable examples from this session
```

Step 1, measure the baseline (15 min). Start the mock CMS in terminal 1. In terminal 2, export CMS_API_KEY=demo-key-123, run python sync_export.py article and python sync_export.py blog_post, and note the times. Read sync_export.py top to bottom: your fetch_page has the same rules.

Step 2, fetch_page (45 min). Port the retry loop to async. Then run python -m pytest test_async_export.py -q -k fetch_page: four tests should pass.

Step 3, fetch_all_entries (45 min). First page, then the other pages in a TaskGroup, all through one semaphore. Run the remaining tests.

Step 4, export and main (30 min). Wire it together, then run the real thing against the mock CMS and compare with the baseline. Restart mock_cms.py before each timing run, or open http://127.0.0.1:8013/stats?reset=1, so the 503s happen again.

Step 5, the experiment (15 min). Run the article export with --max-in-flight 1, 3 and 10, and check /stats after each. Write the three times and the rejected_429 numbers into your PR description with one sentence explaining the result for 10.

```bash
# What a correct run looks like (times differ a little on your laptop)
$ python sync_export.py article
  2623ms WARNING sync_export: HTTP 503 on skip=12, retrying in 0.5s ...
  6139ms INFO sync_export: fetched 30 entries in 6.1s
$ python async_export.py article
  1643ms WARNING async_export: HTTP 503 on skip=12, retrying in 0.5s ...
  2653ms INFO async_export: fetched 30 entries in 2.6s (max 3 in flight)
  2654ms INFO async_export: wrote 30 rows to output/article.csv
$ curl -s http://127.0.0.1:8013/stats
{"requests": 15, "in_flight": 0, "max_in_flight": 3, "rejected_429": 0}
# The failure cases: one ERROR line, the right exit code, no traceback
$ CMS_API_KEY=wrong python async_export.py blog_post; echo $?
   640ms ERROR async_export: HTTP 401: Invalid API key
1
$ python async_export.py product; echo $?
   649ms ERROR async_export: HTTP 422: Content type 'product' was not found
1
```

Acceptance criteria:

- python async_export.py blog_post --out output/posts_async.csv writes a file identical to sync_export.py's output/blog_post.csv: diff prints nothing.

- python async_export.py article finishes in under 3 s (the baseline needs about 6 s) and logs one WARNING retry line for skip=12.

- After that run, /stats shows "max_in_flight": 3 and "rejected_429": 0. With --max-in-flight 1 it shows 1.

- The CSV rows are in uid order, although pages finish in any order, and you did not sort them.

- Wrong key: one ERROR line HTTP 401: Invalid API key, exit code 1, exactly one request in the mock's log. Unknown type: the HTTP 422 line, exit 1. No key set: the SetupError line, exit 2.

- python -m pytest test_async_export.py -q shows 8 passed.

- async_export.py contains no time.sleep and no import requests, opens one AsyncClient per run, and every request goes through the semaphore.

- python -c "import async_export" prints nothing.

- Work on the branch day13, pushed, with a pull request whose description has What, Why, How to test and your Step 5 numbers.

Hints (no solutions, try first):

- Build fetch_page by changing the sync version in three places, not by starting from a blank page. The rules are identical; only the waiting changes.

- Place async with sem: around the await client.get(...) line only. Then check with -v: the DEBUG start and end lines should show three requests overlapping and never four.

- If max_in_flight in /stats is 1 although you use a TaskGroup, look for an await inside the loop that creates the tasks. Creating a task must not wait for the previous one.

- If it is above 3 or you see 429s with the default settings, check where the semaphore is created. There must be exactly one per export run.

- If a test hangs, you are probably awaiting something inside the semaphore block that also needs the semaphore.

- Run the tests after every function. They are faster than the real server and point at the exact rule you broke.

## DSA practice (30 min)

Both problems are in day13/dsa.py with signatures, examples and hints. Both are about counting what is happening at the same time, the question behind every rate limit.

- Max requests in flight. max_in_flight(intervals) takes (start, end) times of requests and returns the largest number open at the same moment. [(0, 5), (1, 3), (2, 6), (7, 8)] gives 3, and [(0, 1), (1, 2)] gives 1 because a request that ends at 1 is closed when the next one starts at 1. Hint: turn each request into two events, +1 at start and -1 at end, sort them so that an end comes before a start at the same time, and keep a running total.

- First rate-limit violation. first_violation(timestamps, limit, window) takes sorted request times and returns the index of the first request that makes more than limit requests in any window of window seconds, or -1. ([0, 0.2, 0.4, 0.5, 1.3], 3, 1.0) gives 3, and ([0, 1, 2, 3], 1, 1.0) gives -1. Hint: keep a deque of recent times; before adding a time t, drop times that are t - window or older from the left.

## Quiz (3 quick questions)

- In Step 5, --max-in-flight 10 was slower than 3. Explain why, using what the server did and what your retry code did.

- A teammate's fetch_page holds the semaphore during await asyncio.sleep(wait) for retries. Nothing crashes. What gets worse, and when would the customer notice?

- A pull request adds r = requests.get(url, timeout=5) inside an async def that runs in a TaskGroup with 20 other tasks. What happens to those tasks while it waits, and what are two ways to fix it?

## What you achieved today

You know why waiting, not computing, is what slows integration code down, and how one event loop keeps many requests in progress on a single thread. You can write and run coroutines, avoid the missing-await trap, run work together with gather and TaskGroup while keeping results in order, and protect a customer's API with a semaphore and timeouts. You used httpx's async client against a slow, rate-limited CMS and tested it with no server at all. These are the same tools you will use for FastAPI handlers, parallel LLM calls and agent tools.

Tomorrow (Day 14): weekly review of Week 2 with scores per area, a mock interview and the Week 3 plan (SQL and Postgres).

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 13 review": your recap answers and check-your-understanding predictions next to the real outputs, async_export.py, the output of python -m pytest test_async_export.py -v, the article and blog_post runs with their times, the /stats output for 1, 3 and 10 in flight, both failure cases, dsa.py, the pull request link and your quiz answers. Say which concept felt least natural; tomorrow's review starts there.

## Recap answers

*Your answers:*
